In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Optimization Solver: Eine Qiskit Function von Q-CTRL Fire Opal
> **Note:** Qiskit Functions sind ein experimentelles Feature, das ausschließlich Nutzerinnen und Nutzern des IBM Quantum&reg; Premium Plan, Flex Plan und On-Prem (über IBM Quantum Platform API) Plan zur Verfügung steht. Sie befinden sich im Preview-Release-Status und können sich noch ändern.

## Überblick
Mit dem Fire Opal Optimization Solver kannst du Optimierungsprobleme im Utility-Scale auf Quantenhardware lösen, ohne Quantenexpertise zu benötigen. Gib einfach die übergeordnete Problemdefinition ein, und der Solver übernimmt den Rest. Der gesamte Workflow ist rauschbewusst und nutzt intern [Fire Opals Performance Management](/guides/q-ctrl-performance-management). Der Solver liefert zuverlässig genaue Lösungen für klassisch anspruchsvolle Probleme, auch im vollen Geräteumfang auf den größten IBM&reg; QPUs.

Der Solver ist flexibel und kann eingesetzt werden, um kombinatorische Optimierungsprobleme zu lösen, die als Zielfunktionen oder beliebige Graphen definiert sind. Probleme müssen nicht auf die Gerätetopologie abgebildet werden. Sowohl unbeschränkte als auch beschränkte Probleme sind lösbar, sofern Einschränkungen als Strafterme formuliert werden können. Die in diesem Leitfaden enthaltenen Beispiele zeigen, wie man ein unbeschränktes und ein beschränktes Optimierungsproblem im Utility-Scale mit verschiedenen Solver-Eingabetypen löst. Das erste Beispiel umfasst ein Max-Cut-Problem, das auf einem 156-Knoten-3-regulären Graphen definiert ist, während das zweite Beispiel ein 50-Knoten-Minimum-Vertex-Cover-Problem angeht, das durch eine Kostenfunktion definiert ist.

Um Zugang zum Optimization Solver zu erhalten, [kontaktiere Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Beschreibung der Function
Der Solver optimiert und automatisiert den gesamten Algorithmus vollständig – von der Fehlerunterdrückung auf Hardwareebene bis hin zu effizientem Problem-Mapping und geschlossener klassischer Optimierung. Im Hintergrund reduziert die Pipeline des Solvers Fehler auf jeder Stufe und ermöglicht so die verbesserte Leistung, die für eine bedeutungsvolle Skalierung erforderlich ist. Der zugrundeliegende Workflow ist vom Quantum Approximate Optimization Algorithm (QAOA) inspiriert, einem hybriden quanten-klassischen Algorithmus. Eine detaillierte Zusammenfassung des vollständigen Optimization-Solver-Workflows findest du im [veröffentlichten Manuskript](https://arxiv.org/abs/2406.01743).

![Visualisierung des Optimization-Solver-Workflows](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

So löst du ein allgemeines Problem mit dem Optimization Solver:
1. Definiere dein Problem als Zielfunktion, einen Graphen oder eine `SparsePauliOp`-Spinkette.
2. Verbinde dich mit der Function über den Qiskit Functions Catalog.
3. Führe das Problem mit dem Solver aus und rufe die Ergebnisse ab.
## Eingaben und Ausgaben
### Eingaben
| Name    | Typ                    | Beschreibung                                                                                                                                                                                         | Erforderlich | Standard | Beispiel |
| ---------  |-------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------| -------- | ---------- | ---------- |
| problem  | `str` oder `SparsePauliOp` | Eine der unter „Akzeptierte Problemformate" aufgeführten Darstellungen                                                                                                                                  | Ja | k. A.   |`Poly(2.0*n[0]*n[1] + 2.0*n[0]*n[2] - 3.13520241298341*n[0] + 2.0*n[1]*n[2] - 3.14469748506794*n[1] - 3.18897660121566*n[2] + 6.0, n[0], n[1], n[2], domain='RR')` |
| problem_type  | `str`                   | Name der Problemklasse; wird nur für Graph- und Spinketten-Problemdefinitionen verwendet, die auf „maxcut" oder „spin_chain" beschränkt sind; für beliebige Zielfunktions-Problemdefinitionen nicht erforderlich | Nein      | `None`| `"maxcut"`      |
| backend_name  | `str`                   | Name des zu verwendenden Backends                                                                                                                                                                          | Nein  | Am wenigsten ausgelastetes Backend, auf das deine Instanz Zugriff hat    | `"ibm_fez"`      |
| options  | `dict`                  | Eingabeoptionen, einschließlich: (Optional) `session_id`: `str`; standardmäßig wird eine neue Session erstellt                                                                                      | Nein | `None`    | `{"session_id": "cw2q70c79ws0008z6g4g"}`     |

**Akzeptierte Problemformate:**
- Polynomielle Ausdrucksdarstellung einer Zielfunktion. Idealerweise in Python mit einem vorhandenen SymPy-Poly-Objekt erstellt und mit [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr) in einen String formatiert.
- Graphdarstellung eines bestimmten Problemtyps. Der Graph sollte mit der networkx-Bibliothek in Python erstellt werden. Er wird dann durch Verwendung der networkx-Funktion `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)` in einen String umgewandelt.
- Spinkettendarstellung eines bestimmten Problems. Die Spinkette sollte als `SparsePauliOp`-Objekt dargestellt werden; weitere Details findest du in der [Dokumentation](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp).

**Unterstützte Backends:**
Führe den folgenden Code aus, um die Liste der aktuell unterstützten Backends anzuzeigen. Wenn dein Gerät nicht aufgeführt ist, [wende dich an Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com), um Support hinzuzufügen.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

service.backends()

[<IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_brisbane')>,
 <IBMBackend('ibm_pittsburgh')>,
 <IBMBackend('ibm_kingston')>,
 <IBMBackend('ibm_torino')>,
 <IBMBackend('ibm_marrakesh')>]

**Options:**
| Name   | Type   | Description  | Default |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | An existing Qiskit Runtime session ID  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | A list of job tags | `[]`|

### Outputs
| Name    | Type | Description | Example |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Solution and metadata listed under "Result dictionary contents"         | `{'solution_bitstring_cost': 3.0, ‘final_bitstring_distribution’: {‘000001’: 100, ‘000011’: 2}, ‘iteration_count’: 3, 'solution_bitstring': ‘000001’,  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |


**Result dictionary contents:**
| Name    | Type | Description |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | The best minimum cost across all iterations        |
| final_bitstring_distribution  | `CountsDict`              | The bitstring counts dictionary associated with the minimum cost        |
| solution_bitstring | `str`              | Bitstring with the best cost in the final distribution         |
| iteration_count  | `int`              | The total number of QAOA iterations performed by the Solver        |
| variables_to_bitstring_index_map  | `float`              | The mapping from the variables to the equivalent bit in the bitstring       |
| best_parameters  | `list[float]`            | The optimized beta and gamma parameters across all iterations  |
| warnings  |`list[str]`              | The warnings produced while compiling or running QAOA (defaults to None)   |

## Benchmarks

[Published benchmarking results](https://arxiv.org/abs/2406.01743) show that the Solver successfully solves problems with over 120 qubits, even outperforming previously published results on quantum annealing and trapped-ion devices. The following benchmark metrics provide a rough indication of the accuracy and scaling of problem types based on a few examples. Actual metrics may differ based on various problem features, such as the number of terms in the objective function (density) and their locality, number of variables, and polynomial order.

The "Number of qubits" indicated is not a hard limitation but represents rough thresholds where you can expect extremely consistent solution accuracy. Larger problem sizes have been successfully solved, and testing beyond these limits is encouraged.

Arbitrary qubit connectivity is supported across all problem types.

| Problem type    | Number of qubits | Example | Accuracy | Total time (s) | Runtime usage (s) | Number of iterations
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Sparsely-connected quadratic problems  | 156 | 3-regular Max-Cut| 100%     | 1764     | 293          | 16 |
| Higher-order binary optimization | 156 | Ising spin-glass model | 100%      | 1461     | 272          | 16 |
| Densely-connected quadratic problems | 50 | Fully-connected Max-Cut| 100%      |  1758    | 268  | 12 |
| Constrained problem with penalty terms | 50 | Weighted Minimum Vertex Cover with 8% edge density | 100%      | 1074     | 215 | 10 |

## Get started

First, authenticate using your [IBM Quantum API key](http://quantum.cloud.ibm.com/). Then, select the Qiskit Function as follows. (This snippet assumes you've already [saved your account](/docs/guides/functions#install-qiskit-functions-catalog-client) to your local environment.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

**Optionen:**
| Name   | Typ   | Beschreibung  | Standard |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | Eine vorhandene Qiskit-Runtime-Session-ID  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | Eine Liste von Job-Tags | `[]`|

### Ausgaben
| Name    | Typ | Beschreibung | Beispiel |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Lösung und Metadaten, aufgeführt unter „Inhalt des Ergebnis-Dictionarys"         | `{'solution_bitstring_cost': 3.0, 'final_bitstring_distribution': {'000001': 100, '000011': 2}, 'iteration_count': 3, 'solution_bitstring': '000001',  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |

**Inhalt des Ergebnis-Dictionarys:**
| Name    | Typ | Beschreibung |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | Die beste minimale Kosten über alle Iterationen hinweg        |
| final_bitstring_distribution  | `CountsDict`              | Das Bitstring-Counts-Dictionary, das den minimalen Kosten zugeordnet ist        |
| solution_bitstring | `str`              | Bitstring mit den besten Kosten in der finalen Verteilung         |
| iteration_count  | `int`              | Die Gesamtzahl der vom Solver durchgeführten QAOA-Iterationen        |
| variables_to_bitstring_index_map  | `float`              | Die Zuordnung der Variablen zum entsprechenden Bit im Bitstring       |
| best_parameters  | `list[float]`            | Die optimierten Beta- und Gamma-Parameter über alle Iterationen hinweg  |
| warnings  |`list[str]`              | Die beim Kompilieren oder Ausführen von QAOA erzeugten Warnungen (Standard: None)   |
## Benchmarks
[Veröffentlichte Benchmarking-Ergebnisse](https://arxiv.org/abs/2406.01743) zeigen, dass der Solver Probleme mit über 120 Qubits erfolgreich löst und dabei sogar zuvor veröffentlichte Ergebnisse auf Quanten-Annealing- und Trapped-Ion-Geräten übertrifft. Die folgenden Benchmark-Metriken geben einen groben Hinweis auf die Genauigkeit und Skalierung von Problemtypen anhand einiger Beispiele. Die tatsächlichen Metriken können je nach verschiedenen Problemeigenschaften variieren, wie z. B. der Anzahl der Terme in der Zielfunktion (Dichte) und deren Lokalität, der Anzahl der Variablen und der polynomiellen Ordnung.

Die angegebene „Anzahl der Qubits" ist keine absolute Begrenzung, sondern stellt ungefähre Schwellenwerte dar, bei denen du eine äußerst konsistente Lösungsgenauigkeit erwarten kannst. Größere Problemgrößen wurden erfolgreich gelöst, und Tests jenseits dieser Grenzen sind ausdrücklich erwünscht.

Beliebige Qubit-Konnektivität wird für alle Problemtypen unterstützt.

| Problemtyp    | Anzahl der Qubits | Beispiel | Genauigkeit | Gesamtzeit (s) | Runtime-Nutzung (s) | Anzahl der Iterationen
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Dünn verbundene quadratische Probleme  | 156 | 3-reguläres Max-Cut | 100%     | 1764     | 293          | 16 |
| Optimierung binärer Probleme höherer Ordnung | 156 | Ising-Spinglas-Modell | 100%      | 1461     | 272          | 16 |
| Dicht verbundene quadratische Probleme | 50 | Vollständig verbundenes Max-Cut | 100%      |  1758    | 268  | 12 |
| Beschränktes Problem mit Straftermen | 50 | Gewichtetes Minimum Vertex Cover mit 8 % Kantendichte | 100%      | 1074     | 215 | 10 |
## Erste Schritte
Authentifiziere dich zunächst mit deinem [IBM Quantum API-Schlüssel](http://quantum.cloud.ibm.com/). Wähle dann die Qiskit Function wie folgt aus. (Dieser Codeausschnitt setzt voraus, dass du dein Konto bereits [in deiner lokalen Umgebung gespeichert hast](/guides/functions#install-qiskit-functions-catalog-client).)

In [2]:
# %pip install networkx numpy

## Beispiel: Unbeschränkte Optimierung
Führe das [Maximum-Cut](https://en.wikipedia.org/wiki/Maximum_cut)-Problem (Max-Cut) aus. Das folgende Beispiel demonstriert die Fähigkeiten des Solvers an einem 156-Knoten-3-regulären ungewichteten Graphen-Max-Cut-Problem, du kannst aber auch gewichtete Graphenprobleme lösen.
Zusätzlich zu `qiskit-ibm-catalog` verwendest du in diesem Beispiel auch die folgenden Pakete: `networkx` und `numpy`. Du kannst diese Pakete installieren, indem du die folgende Zelle auskommentierst, wenn du dieses Beispiel in einem Notebook mit dem IPython-Kernel ausführst.

In [2]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [3]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

The Solver accepts a string as the problem definition input.

In [4]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif)

Der Solver akzeptiert einen String als Problemdefinitions-Eingabe.

In [9]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [ ]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [14]:
# Get job status
print(maxcut_job.status())

QUEUED


Überprüfe den [Status](/guides/functions#check-job-status) deiner Qiskit-Function-Arbeitslast oder rufe [Ergebnisse](/guides/functions#retrieve-results) wie folgt ab:

In [15]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 209.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior Max-Cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [ ]:
# %pip install numpy networkx sympy

### 3. Das Ergebnis abrufen
Rufe den optimalen Schnittwert aus dem Ergebnis-Dictionary ab.

> **Note:** Die Zuordnung der Variablen zum Bitstring kann sich geändert haben. Das Ausgabe-Dictionary enthält ein `variables_to_bitstring_index_map`-Unter-Dictionary, das hilft, die Reihenfolge zu überprüfen.

In [ ]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

A standard optimization model for weighted MVC can be formulated as follows. First, a penalty must be added for any case where an edge is not connected to a vertex in the subset. Therefore, let $n_i = 1$ if vertex $i$ is in the cover (i.e., in the subset) and $n_i = 0$ otherwise. Second, the goal is to minimize the total number of vertices in the subset, which can be represented by the following function:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

Du kannst die Genauigkeit des Ergebnisses überprüfen, indem du das Problem klassisch mit Open-Source-Solvern wie [PuLP](https://coin-or.github.io/pulp/) löst, sofern der Graph nicht dicht verbunden ist. Bei Problemen mit hoher Dichte sind möglicherweise fortgeschrittene klassische Solver erforderlich, um die Lösung zu validieren.
## Beispiel: Beschränkte Optimierung
Das vorherige Max-Cut-Beispiel ist ein klassisches quadratisches unbeschränktes binäres Optimierungsproblem. Q-CTRLs Optimization Solver kann für verschiedene Problemtypen eingesetzt werden, einschließlich beschränkter Optimierung. Du kannst beliebige Problemtypen lösen, indem du die Problemdefinition als Polynom eingibst, bei dem Einschränkungen als Strafterme modelliert werden.

Das folgende Beispiel zeigt, wie eine Kostenfunktion für ein beschränktes Optimierungsproblem, das [Minimum Vertex Cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC), konstruiert wird.
Zusätzlich zu den Paketen `qiskit-ibm-catalog` und `qiskit` verwendest du in diesem Beispiel auch die folgenden Pakete: `numpy`, `networkx` und `sympy`. Du kannst diese Pakete installieren, indem du die folgende Zelle auskommentierst, wenn du dieses Beispiel in einem Notebook mit dem IPython-Kernel ausführst.

In [ ]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

### 1. Das Problem definieren
Definiere ein zufälliges MVC-Problem, indem du einen Graphen mit zufällig gewichteten Knoten erzeugst.

In [ ]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif)

Ein Standardoptimierungsmodell für gewichtetes MVC kann wie folgt formuliert werden. Zunächst muss eine Strafe für jeden Fall hinzugefügt werden, in dem eine Kante nicht mit einem Knoten in der Teilmenge verbunden ist. Sei daher $n_i = 1$, wenn Knoten $i$ zur Überdeckung gehört (d. h. in der Teilmenge ist), und $n_i = 0$ andernfalls. Das Ziel ist es, die Gesamtzahl der Knoten in der Teilmenge zu minimieren, was durch die folgende Funktion dargestellt werden kann:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
print(mvc_job.status())

Jetzt sollte jede Kante im Graphen mindestens einen Endpunkt der Überdeckung enthalten, was als Ungleichung ausgedrückt werden kann:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Jeder Fall, in dem eine Kante nicht mit dem Knoten der Überdeckung verbunden ist, muss bestraft werden. Dies kann in der Kostenfunktion durch Hinzufügen einer Strafe der Form $P(1-n_i-n_j+n_i n_j)$ dargestellt werden, wobei $P$ eine positive Strafkonstante ist. Damit ist eine unbeschränkte Alternative zur beschränkten Ungleichung für gewichtetes MVC:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [ ]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


### 2. Das Problem ausführen